In [1]:
# ============================================================================
# MULTI-DATASET TRAINING: 9 CLASSES FROM 3 DATASETS
# Alzheimer (4) + Chest (2) + Lung (3) = 9 classes total


In [2]:
!pip install -q timm torch torchvision tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import time
import json
from pathlib import Path
import timm
import matplotlib.pyplot as plt
from PIL import Image

print("✓ All packages imported successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ All packages imported successfully!


In [ ]:
# CONFIGURATION - MULTI-DATASET


In [3]:
# 🔧 UPDATE THESE PATHS TO YOUR DATASET LOCATIONS
DATA_PATHS = {
    'alzheimer': "/content/drive/MyDrive/FYP/datasets/Alzheimer's MRI",
    'chest': "/content/drive/MyDrive/FYP/datasets/Chest",
    'lung': "/content/drive/MyDrive/FYP/datasets/Lung"
}

OUTPUT_DIR = '/content/drive/MyDrive/FYP/datasets/output_dir'

# Class mapping for unified label space (0-8)
CLASS_MAPPING = {

    # Alzheimer classes (0-3)
    'Mild Impairment': 0,
    'Moderate Impairment': 1,
    'No Impairment': 2,
    'Very Mild Impairment': 3,

    # Chest classes (4-5)
    'NORMAL': 4,
    'PNEUMONIA': 5,

    # Lung classes (6-8)
    'Bengin cases': 6,
    'Malignant cases': 7,
    'Normal cases': 8
}

# Reverse mapping for predictions
ID_TO_CLASS = {v: k for k, v in CLASS_MAPPING.items()}

# Dataset prefixes for better tracking
DATASET_PREFIX = {
    'alzheimer': 'ALZ',
    'chest': 'CHEST',
    'lung': 'LUNG'
}

# Training Configuration (OPTIMIZED FOR SPEED)
# ============================================================================
# CONFIGURATION - MULTI-DATASET (ULTRA-OPTIMIZED)
# ============================================================================

# Training Configuration (MAXIMUM SPEED)
CONFIG = {
    'batch_size': 128,         # ✅ QUADRUPLED (was 32)
    'num_epochs': 50,
    'learning_rate': 4e-4,     # ✅ QUADRUPLED (because batch quadrupled)
    'model_size': 'tiny',
    'image_size': 224,
    'train_split': 0.8,
    'seed': 42,
    'num_workers': 4,          # ✅ DOUBLED (critical!)
    'early_stopping_patience': 10,
    'num_classes': 9
}



os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Configuration set!")
print(f"\n📁 Dataset paths:")
for name, path in DATA_PATHS.items():
    print(f"  {name}: {path}")
print(f"\n💾 Output: {OUTPUT_DIR}")
print(f"🎯 Total classes: {CONFIG['num_classes']}")
print(f"📊 Class mapping:")
for cls_name, cls_id in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]):
    print(f"  [{cls_id}] {cls_name}")


✓ Configuration set!

📁 Dataset paths:
  alzheimer: /content/drive/MyDrive/FYP/datasets/Alzheimer's MRI
  chest: /content/drive/MyDrive/FYP/datasets/Chest
  lung: /content/drive/MyDrive/FYP/datasets/Lung

💾 Output: /content/drive/MyDrive/FYP/datasets/output_dir
🎯 Total classes: 9
📊 Class mapping:
  [0] Mild Impairment
  [1] Moderate Impairment
  [2] No Impairment
  [3] Very Mild Impairment
  [4] NORMAL
  [5] PNEUMONIA
  [6] Bengin cases
  [7] Malignant cases
  [8] Normal cases


In [4]:
# VERIFY DATASET PATHS


In [5]:
print("\n" + "="*70)
print("VERIFYING DATASET PATHS")
print("="*70)

total_images = 0
dataset_stats = {}

for dataset_name, data_root in DATA_PATHS.items():
    print(f"\n📂 {dataset_name.upper()} Dataset: {data_root}")

    if os.path.exists(data_root):
        print(f"   ✅ Path exists")

        classes = [d for d in os.listdir(data_root)
                   if os.path.isdir(os.path.join(data_root, d))]

        if classes:
            dataset_images = 0
            print(f"   ✅ Found {len(classes)} classes:")
            for cls in sorted(classes):
                class_path = os.path.join(data_root, cls)
                num_images = len([f for f in os.listdir(class_path)
                                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])

                # Get mapped class ID
                mapped_id = CLASS_MAPPING.get(cls, "UNKNOWN")
                print(f"      [{mapped_id}] {cls:30s}: {num_images:5d} images")
                dataset_images += num_images

            dataset_stats[dataset_name] = {'classes': len(classes), 'images': dataset_images}
            total_images += dataset_images
        else:
            print(f"   ❌ No class folders found!")
    else:
        print(f"   ❌ Path does NOT exist!")
        print(f"\n⚠️ Please update DATA_PATHS['{dataset_name}']")

print(f"\n{'='*70}")
print(f"📊 TOTAL: {total_images} images across {len(DATA_PATHS)} datasets")
for ds_name, stats in dataset_stats.items():
    print(f"   {ds_name}: {stats['classes']} classes, {stats['images']} images")
print("="*70)



VERIFYING DATASET PATHS

📂 ALZHEIMER Dataset: /content/drive/MyDrive/FYP/datasets/Alzheimer's MRI
   ✅ Path exists
   ✅ Found 4 classes:
      [0] Mild Impairment               :  2739 images
      [1] Moderate Impairment           :  2572 images
      [2] No Impairment                 :  3200 images
      [3] Very Mild Impairment          :  3008 images

📂 CHEST Dataset: /content/drive/MyDrive/FYP/datasets/Chest
   ✅ Path exists
   ✅ Found 2 classes:
      [4] NORMAL                        :  3166 images
      [5] PNEUMONIA                     :  8543 images

📂 LUNG Dataset: /content/drive/MyDrive/FYP/datasets/Lung
   ✅ Path exists
   ✅ Found 4 classes:
      [6] Bengin cases                  :   120 images
      [7] Malignant cases               :   561 images
      [8] Normal cases                  :   416 images
      [UNKNOWN] Test cases                    :   197 images

📊 TOTAL: 24522 images across 3 datasets
   alzheimer: 4 classes, 11519 images
   chest: 2 classes, 11709 imag

In [6]:
# ============================================================================
# SPEED OPTIMIZATION: COPY DATA TO LOCAL STORAGE (25x FASTER!)
# ============================================================================

import shutil
import time

print("="*70)
print("🚀 COPYING DATASETS TO LOCAL STORAGE")
print("   (Takes 2-3 min but saves 45 min per epoch!)")
print("="*70)

LOCAL_BASE = "/content/local_data"
LOCAL_DATA_PATHS = {}

for dataset_name, drive_path in DATA_PATHS.items():
    local_path = os.path.join(LOCAL_BASE, dataset_name, "train")
    LOCAL_DATA_PATHS[dataset_name] = local_path

    if os.path.exists(drive_path):
        print(f"\n📦 Copying {dataset_name}...")
        start = time.time()

        # Remove if exists
        if os.path.exists(local_path):
            shutil.rmtree(os.path.dirname(local_path))

        # Copy directory
        shutil.copytree(drive_path, local_path)

        # Count files
        total_files = sum(len(files) for _, _, files in os.walk(local_path))
        elapsed = time.time() - start
        print(f"   ✅ {total_files} files copied in {elapsed:.1f}s")
    else:
        print(f"   ⚠️ {drive_path} not found!")

# ✅ UPDATE DATA_PATHS TO USE LOCAL COPIES
DATA_PATHS = LOCAL_DATA_PATHS

print("\n" + "="*70)
print("✅ ALL DATA NOW ON FAST LOCAL STORAGE!")
print("="*70)
print("\n📁 Updated paths:")
for name, path in DATA_PATHS.items():
    print(f"  {name}: {path}")


🚀 COPYING DATASETS TO LOCAL STORAGE
   (Takes 2-3 min but saves 45 min per epoch!)

📦 Copying alzheimer...
   ✅ 11519 files copied in 141.3s

📦 Copying chest...
   ✅ 11714 files copied in 149.5s

📦 Copying lung...
   ✅ 1295 files copied in 9.2s

✅ ALL DATA NOW ON FAST LOCAL STORAGE!

📁 Updated paths:
  alzheimer: /content/local_data/alzheimer/train
  chest: /content/local_data/chest/train
  lung: /content/local_data/lung/train


In [ ]:
# CUSTOM DATASET CLASS


In [7]:
class MultiDatasetFolder(Dataset):
    """Custom dataset that loads images from multiple datasets with unified labels"""

    def __init__(self, data_paths, class_mapping, transform=None):
        self.transform = transform
        self.class_mapping = class_mapping
        self.samples = []

        # Load all images from all datasets
        for dataset_name, root_path in data_paths.items():
            if not os.path.exists(root_path):
                print(f"⚠️ Skipping {dataset_name}: path not found")
                continue

            classes = [d for d in os.listdir(root_path)
                      if os.path.isdir(os.path.join(root_path, d))]

            for cls in classes:
                if cls not in class_mapping:
                    print(f"⚠️ Class '{cls}' not in mapping, skipping")
                    continue

                class_path = os.path.join(root_path, cls)
                class_id = class_mapping[cls]

                # Get all images in this class folder
                for img_name in os.listdir(class_path):
                  # Skip macOS hidden files
                  if img_name.startswith('._'):
                      continue
                  if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                      img_path = os.path.join(class_path, img_name)
                      self.samples.append((img_path, class_id, dataset_name))

        print(f"\n✓ Loaded {len(self.samples)} total images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, dataset_name = self.samples[idx]

        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a black image if loading fails
            if self.transform:
                return self.transform(Image.new('RGB', (224, 224))), label
            return Image.new('RGB', (224, 224)), label


In [8]:
# DATA AUGMENTATION & LOADING


In [9]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

# Transforms
train_transform = transforms.Compose([
    transforms.Resize(CONFIG['image_size'] + 32, InterpolationMode.BICUBIC),
    transforms.RandomCrop(CONFIG['image_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(CONFIG['image_size'] + 32, InterpolationMode.BICUBIC),
    transforms.CenterCrop(CONFIG['image_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create full dataset
print("\n" + "="*70)
print("LOADING DATASETS")
print("="*70)

full_dataset = MultiDatasetFolder(
    data_paths=DATA_PATHS,
    class_mapping=CLASS_MAPPING,
    transform=train_transform
)

# Split into train and validation
train_size = int(CONFIG['train_split'] * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(CONFIG['seed'])
)

# Update val_dataset transform
val_dataset.dataset.transform = val_transform

print(f"\n✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")

# Create dataloaders
# Create OPTIMIZED dataloaders
print("\n" + "="*70)
print("CREATING ULTRA-OPTIMIZED DATALOADERS")
print("="*70)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],  # 8 workers
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,           # ✅ DOUBLED (was 2)
    drop_last=True               # ✅ Avoid small last batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'] * 2,  # 256 for validation
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,           # ✅ DOUBLED
    drop_last=False
)

print(f"✓ Train loader:")
print(f"  - Batch size: {CONFIG['batch_size']}")
print(f"  - Workers: {CONFIG['num_workers']}")
print(f"  - Prefetch factor: 4")
print(f"  - Total batches: {len(train_loader)}")
print(f"✓ Val loader:")
print(f"  - Batch size: {CONFIG['batch_size'] * 2}")
print(f"  - Total batches: {len(val_loader)}")
print("="*70)


print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")



LOADING DATASETS
⚠️ Class 'Test cases' not in mapping, skipping

✓ Loaded 18477 total images

✓ Train samples: 14781
✓ Val samples: 3696

CREATING ULTRA-OPTIMIZED DATALOADERS
✓ Train loader:
  - Batch size: 128
  - Workers: 4
  - Prefetch factor: 4
  - Total batches: 115
✓ Val loader:
  - Batch size: 256
  - Total batches: 15
✓ Train batches: 115
✓ Val batches: 15


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [10]:
# MODEL ARCHITECTURE


In [11]:
class ConvNeXtV2FeatureExtractor(nn.Module):
    """ConvNeXt-v2 with 512-D L2-normalized features"""

    def __init__(self, model_size='tiny', num_classes=9, feature_dim=512):
        super().__init__()

        # Load pretrained ConvNeXt-v2
        model_name = f'convnextv2_{model_size}.fcmae_ft_in22k_in1k'
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)

        # Get backbone output dimension
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            backbone_dim = self.backbone(dummy_input).shape[1]

        # Projection head with GELU and L2 normalization
        self.projection = nn.Sequential(
            nn.Linear(backbone_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU()  # ← Added activation
        )

        # Classification head
        self.classifier = nn.Linear(feature_dim, num_classes)

        self.feature_dim = feature_dim
        self.num_classes = num_classes

    def forward(self, x, return_features=False):
        # Backbone
        x = self.backbone(x)

        # Project to 512-D
        features = self.projection(x)

        # L2 normalize
        features = nn.functional.normalize(features, p=2, dim=1)

        if return_features:
            return features

        # Classify
        logits = self.classifier(features)
        return logits


In [12]:
# TRAINING SETUP


In [13]:
# ============================================================================
# TRAINING SETUP (ALL OPTIMIZATIONS ENABLED)
# ============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")

# ✅ Enable ALL performance optimizations
torch.backends.cudnn.benchmark = True           # Auto-tune convolutions
torch.backends.cudnn.deterministic = False      # Allow non-deterministic (faster)
torch.backends.cuda.matmul.allow_tf32 = True    # Use TF32 on Ampere GPUs
torch.backends.cudnn.allow_tf32 = True          # Use TF32 for cuDNN

print("✓ All PyTorch optimizations enabled:")
print(f"  - cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"  - cuDNN deterministic: {torch.backends.cudnn.deterministic}")
print(f"  - TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"  - TF32 cuDNN: {torch.backends.cudnn.allow_tf32}")

model = ConvNeXtV2FeatureExtractor(
    model_size=CONFIG['model_size'],
    num_classes=CONFIG['num_classes'],
    feature_dim=512
).to(device)

# Use fused optimizer (faster)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=0.05,
    fused=True  # ✅ Fused optimizer (10-20% faster)
)
scaler = GradScaler()

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['num_epochs'],
    eta_min=1e-6
)

print(f"✓ Model: ConvNeXt-v2-{CONFIG['model_size']}")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"✓ Fused optimizer: True")



✓ Using device: cuda
✓ All PyTorch optimizations enabled:
  - cuDNN benchmark: True
  - cuDNN deterministic: False
  - TF32 matmul: True
  - TF32 cuDNN: True


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in

✓ Model: ConvNeXt-v2-tiny
✓ Parameters: 28,265,865
✓ Fused optimizer: True


/tmp/ipython-input-4060388957.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [14]:
import torch
torch.cuda.is_available()


True

In [15]:
# ============================================================================
# TRAINING LOOP (FIXED)
# ============================================================================

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from torch.cuda.amp import autocast, GradScaler

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Training', ncols=100)
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if batch_idx % 10 == 0:
            pbar.set_postfix({
                'loss': f'{running_loss/(batch_idx+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation', ncols=100)
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({
                'loss': f'{running_loss/(pbar.n+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Training loop
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

start_time = time.time()

for epoch in range(CONFIG['num_epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print("-" * 70)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Update scheduler
    scheduler.step()

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        patience_counter = 0

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_mapping': CLASS_MAPPING,
            'id_to_class': ID_TO_CLASS
        }, os.path.join(OUTPUT_DIR, 'best_model.pt'))

        print(f"✓ Best model saved! (Val Acc: {val_acc:.4f})")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= CONFIG['early_stopping_patience']:
        print(f"\n⏹ Early stopping triggered after {epoch+1} epochs")
        break

elapsed = time.time() - start_time

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Total Time: {elapsed/60:.2f} minutes")
print(f"Best Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")
print(f"Final Train Acc: {train_acc:.4f}")
print(f"Final Val Acc: {val_acc:.4f}")



STARTING TRAINING

Epoch 1/50
----------------------------------------------------------------------


Training:   0%|                                                             | 0/115 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Validation: 100%|██████████████████████████| 15/15 [01:05<00:00,  4.38s/it, loss=1.2679, acc=83.12%]



Train Loss: 1.6323 | Train Acc: 0.7238
Val Loss: 1.2679 | Val Acc: 0.8312
LR: 4.00e-04
✓ Best model saved! (Val Acc: 0.8312)

Epoch 2/50
----------------------------------------------------------------------


Validation: 100%|██████████████████████████| 15/15 [00:36<00:00,  2.44s/it, loss=0.7604, acc=88.96%]



Train Loss: 0.9652 | Train Acc: 0.8798
Val Loss: 0.7604 | Val Acc: 0.8896
LR: 3.98e-04
✓ Best model saved! (Val Acc: 0.8896)

Epoch 3/50
----------------------------------------------------------------------


...


Train Loss: 0.0074 | Train Acc: 0.9999
Val Loss: 0.0593 | Val Acc: 0.9881
LR: 1.16e-04

Epoch 33/50
----------------------------------------------------------------------


Validation: 100%|██████████████████████████| 15/15 [00:36<00:00,  2.43s/it, loss=0.0578, acc=98.81%]



Train Loss: 0.0071 | Train Acc: 0.9999
Val Loss: 0.0578 | Val Acc: 0.9881
LR: 1.04e-04

Epoch 34/50
----------------------------------------------------------------------


Validation: 100%|██████████████████████████| 15/15 [00:35<00:00,  2.34s/it, loss=0.0596, acc=98.78%]


Train Loss: 0.0068 | Train Acc: 0.9999
Val Loss: 0.0596 | Val Acc: 0.9878
LR: 9.36e-05

⏹ Early stopping triggered after 34 epochs

TRAINING COMPLETE
Total Time: 118.22 minutes
Best Val Accuracy: 0.9884 (Epoch 24)
Final Train Acc: 0.9999
Final Val Acc: 0.9878


In [ ]:
# EXPORT FEATURE EXTRACTOR


In [21]:
print("\n" + "="*70)
print("EXPORTING FEATURE EXTRACTOR")
print("="*70)

# Load best model
checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")

# Test feature extraction
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 224, 224).to(device)
    features = model(dummy_input, return_features=True)

    print(f"\n✓ Feature extraction test:")
    print(f"  Input shape: {dummy_input.shape}")
    print(f"  Output shape: {features.shape}")
    print(f"  Feature dimension: {features.shape[1]}")

    # Check L2 normalization
    norms = torch.norm(features, p=2, dim=1).cpu().numpy()
    print(f"  L2 norms: {norms.tolist()[:3]} (should be ~1.0)")

    if np.allclose(norms, 1.0, atol=1e-6):
        print(f"  ✅ Features are L2-normalized!")
    else:
        print(f"  ⚠️ Warning: Features may not be properly normalized")

# Save feature extractor
feature_extractor_path = os.path.join(OUTPUT_DIR, 'convnextv2_multi_512d.pt')
torch.save({
    'model': model,
    'model_state_dict': model.state_dict(),
    'feature_dim': 512,
    'num_classes': CONFIG['num_classes'],
    'class_mapping': CLASS_MAPPING,
    'id_to_class': ID_TO_CLASS,
    'best_val_acc': best_val_acc,
    'config': CONFIG
}, feature_extractor_path)

file_size = os.path.getsize(feature_extractor_path) / (1024**2)
print(f"\n✓ Saved feature extractor: {feature_extractor_path}")
print(f"  File size: {file_size:.2f} MB")

print("\n" + "="*70)
print("EXPORT COMPLETE")
print("="*70)

print(f"\n✅ Architecture verification:")
print(f"  Input: 224×224×3")
print(f"  Backbone: ConvNeXt-v2 (ImageNet-22K pretrained)")
print(f"  Global Avg Pool: ✓")
print(f"  Linear projection: ✓")
print(f"  LayerNorm: ✓")
print(f"  GELU: ✓")
print(f"  L2 Normalization: ✓")
print(f"  Output: 512-D semantic feature vectors")
print(f"  Classification: {CONFIG['num_classes']} classes")



EXPORTING FEATURE EXTRACTOR
✓ Loaded best model from epoch 24

✓ Feature extraction test:
  Input shape: torch.Size([2, 3, 224, 224])
  Output shape: torch.Size([2, 512])
  Feature dimension: 512
  L2 norms: [1.0, 1.0] (should be ~1.0)
  ✅ Features are L2-normalized!

✓ Saved feature extractor: /content/drive/MyDrive/FYP/datasets/output_dir/convnextv2_multi_512d.pt
  File size: 107.99 MB

EXPORT COMPLETE

✅ Architecture verification:
  Input: 224×224×3
  Backbone: ConvNeXt-v2 (ImageNet-22K pretrained)
  Global Avg Pool: ✓
  Linear projection: ✓
  LayerNorm: ✓
  GELU: ✓
  L2 Normalization: ✓
  Output: 512-D semantic feature vectors
  Classification: 9 classes


In [17]:
# INFERENCE EXAMPLE


In [1]:
# ============================================================================
# INFERENCE EXAMPLE (COMPLETE FIXED VERSION)
# ============================================================================

import torch.nn.functional as F

print("\n" + "="*70)
print("INFERENCE EXAMPLE")
print("="*70)

# Define model class first (required for loading)
class ConvNeXtV2FeatureExtractor(nn.Module):
    """ConvNeXt-v2 with 512-D L2-normalized features"""

    def __init__(self, model_size='tiny', num_classes=9, feature_dim=512):
        super().__init__()

        model_name = f'convnextv2_{model_size}.fcmae_ft_in22k_in1k'
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)

        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            backbone_dim = self.backbone(dummy_input).shape[1]

        self.projection = nn.Sequential(
            nn.Linear(backbone_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU()
        )

        self.classifier = nn.Linear(feature_dim, num_classes)
        self.feature_dim = feature_dim
        self.num_classes = num_classes

    def forward(self, x, return_features=False):
        x = self.backbone(x)
        features = self.projection(x)
        features = nn.functional.normalize(features, p=2, dim=1)

        if return_features:
            return features

        logits = self.classifier(features)
        return logits

# Load feature extractor (with weights_only=False since we trust our model)
checkpoint = torch.load(feature_extractor_path, weights_only=False)
loaded_model = checkpoint['model']
loaded_model.to(device)
loaded_model.eval()

print(f"\n✓ Loaded feature extractor")
print(f"  Model: ConvNeXt-v2-{checkpoint['config']['model_size']}")
print(f"  Classes: {checkpoint['config']['num_classes']}")
print(f"  Best accuracy: {checkpoint['best_val_acc']:.4f}")

# Print class mapping
print(f"\n📋 Class Mapping:")
class_mapping = checkpoint['class_mapping']
id_to_class = checkpoint['id_to_class']

# Convert to integer keys
id_to_class_int = {int(k): v for k, v in id_to_class.items()}

for cls_id, cls_name in sorted(id_to_class_int.items()):
    print(f"  [{cls_id}] {cls_name}")

# Extract features from validation batch
val_iter = iter(val_loader)
images, labels = next(val_iter)
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    features = loaded_model(images, return_features=True)
    logits = loaded_model(images)
    predictions = torch.argmax(logits, dim=1)

print(f"\n✓ Extracted features from validation batch:")
print(f"  Batch size: {features.shape[0]}")
print(f"  Feature shape: {features.shape}")
print(f"  Feature range: [{features.min().item():.3f}, {features.max().item():.3f}]")

# Check L2 norms
norms = torch.norm(features, p=2, dim=1).cpu().numpy()
print(f"  L2 norms (sample): {norms[:5].tolist()}")
print(f"  All L2-normalized: {np.allclose(norms, 1.0, atol=1e-6)}")

# Example cosine similarity
if len(features) >= 2:
    cos_sim = F.cosine_similarity(features[0:1], features[1:2]).item()
    print(f"\n✓ Example cosine similarity (img1 vs img2): {cos_sim:.4f}")

# Example predictions
print(f"\n✓ Sample predictions:")

for i in range(min(5, len(predictions))):
    pred_idx = predictions[i].item()
    true_idx = labels[i].item()

    # Get class names
    pred_class = id_to_class_int.get(pred_idx, f"Class-{pred_idx}")
    true_class = id_to_class_int.get(true_idx, f"Class-{true_idx}")

    match = "✓" if pred_idx == true_idx else "✗"

    # Show confidence
    probs = F.softmax(logits[i], dim=0)
    confidence = probs[pred_idx].item()

    print(f"  [{i+1}] Pred: {pred_class:20s} | True: {true_class:20s} | {match} | Conf: {confidence:.2%}")

# Overall accuracy for this batch
correct = (predictions == labels).sum().item()
total = len(labels)
batch_acc = correct / total

print(f"\n✓ Batch Accuracy: {correct}/{total} = {batch_acc:.2%}")

# Per-class accuracy
print(f"\n📊 Per-class samples in batch:")
unique_labels, counts = torch.unique(labels, return_counts=True)
for label_idx, count in zip(unique_labels, counts):
    class_name = id_to_class_int.get(label_idx.item(), f"Class-{label_idx.item()}")
    class_correct = ((predictions == label_idx) & (labels == label_idx)).sum().item()
    class_acc = class_correct / count.item()
    print(f"  {class_name:20s}: {class_correct:3d}/{count.item():3d} = {class_acc:.1%}")

print("\n" + "="*70)
print("✅ INFERENCE COMPLETE!")
print("="*70)



INFERENCE EXAMPLE


NameError: name 'nn' is not defined